# Pairwise metric comparison
One value per method from the benchmark tables (cohort means). Baselines are left out of correlations.

In [1]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import font_manager

sys.path.insert(0, str(Path.cwd().parents[0]))
from cellseg_benchmark import BASE_PATH
from cellseg_benchmark._constants import main_methods, method_colors, method_names, metric_names, methods_3D
from cellseg_benchmark.metrics import clean_method_name

font_manager.fontManager.addfont(f"{BASE_PATH}/misc/Arial.ttf")
plt.rcParams.update({"font.family": "Arial", "svg.fonttype": "none", "figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})
cohorts = ["aging", "ABCAtlas", "VizgenMouseBrain"]
metrics_dir = Path(BASE_PATH) / "metrics"

In [ ]:
tables = {c: pd.read_csv(metrics_dir / "merged/benchmark_tables/all" / f"{c}.csv", index_col=0) for c in [*cohorts, "combined"]}
table = tables["combined"]
cohort_names = {"aging": "Aging", "ABCAtlas": "ABC Atlas", "VizgenMouseBrain": "Vizgen"}
names = metric_names | {"low_z_integrity_vs_tissue": "Z integrity"}
vals = table.filter(like="_raw").assign(low_z_integrity_vs_tissue_raw=lambda d: -d.low_z_integrity_vs_tissue_raw).rename(columns=lambda c: names.get(c.removesuffix("_raw"), c))
raw = {v: k for k, v in method_names.items()}
is3d = lambda ids: pd.Index([raw[m] for m in ids]).str.startswith(tuple(methods_3D))
is_baseline = lambda ids: pd.Index(ids).str.startswith(("Raster", "Voronoi"))
grey = {True: "#3d3d3d", False: "#a3a3a3"}
out = metrics_dir / "merged/plots"
out.mkdir(parents=True, exist_ok=True)

In [ ]:
d = vals[~is_baseline(vals.index)]
g = sns.clustermap(d.corr("spearman").fillna(0), cmap="RdBu_r", vmin=-1, vmax=1, annot=True, fmt=".1f", annot_kws={"size": 5},
                   figsize=(3, 3), dendrogram_ratio=0.05, cbar_pos=None, linewidths=0)
g.ax_heatmap.set(xlabel="", ylabel="")
g.ax_heatmap.tick_params(labelsize=6, length=0)
g.savefig(out / "metric_correlation_table.png", dpi=600, bbox_inches="tight", transparent=True)
print(len(d), "methods,", d.notna().sum().min(), "with all metrics")

In [ ]:
plt.rcParams.update({"font.size": 7, "axes.linewidth": 0.5,
                     "xtick.major.width": 0.5, "ytick.major.width": 0.5, "xtick.major.size": 2, "ytick.major.size": 2})
labelled = {  # label, offset (pt), alignment
    "Cellpose1 3D P VPT": ("Cellpose1\n3D P VPT", -8, -16, "center"),
    "Proseg 3D (CP1 3D N)": ("Proseg 3D\n(CP1 3D N)", 8, 9, "left"),
    "Cellpose1 P VPT": ("Cellpose1\nP VPT", 8, 9, "left"),
    "Baysor": ("Baysor", -10, -3, "right"),
}
pairs = {
    "Proseg": {"Cellpose1 3D P VPT": "Proseg 3D (CP1 3D P)", "Cellpose1 3D N VPT": "Proseg 3D (CP1 3D N)",
               "Cellpose1 3D P+N VPT": "Proseg 3D (CP1 3D P+N)", "Cellpose1 N Sopa": "Proseg 3D (CP1 N)",
               "Cellpose1 P Sopa": "Proseg 3D (CP1 P)", "Cellpose2 P Sopa": "Proseg 3D (CP2 P)"},
    "Baysor": {"Cellpose1 N Sopa": "Baysor Sopa (CP1 N, 1.0)", "Cellpose1 P Sopa": "Baysor Sopa (CP1 P, 0.2)",
               "Cellpose2 P Sopa": "Baysor Sopa (CP2 P, 0.2)"},
}

d = table.loc[[m for m in main_methods if m in table.index], ["pct_assigned_qced_raw", "marker_f1_raw"]].dropna()
d.columns = ["x", "y"]
d["x"] *= 100
d["3d"] = is3d(d.index)

fig, (ax, af) = plt.subplots(1, 2, figsize=(3.8, 1.55), sharey=True, gridspec_kw={"width_ratios": [3.4, 1], "wspace": 0.12})
for k in (False, True):
    ax.scatter(d.x[d["3d"] == k], d.y[d["3d"] == k], s=15, c=grey[k], lw=0, label="3D" if k else "2D")
for n, (text, dx, dy, ha) in labelled.items():
    ax.annotate(text, (d.x[n], d.y[n]), xytext=(dx, dy), textcoords="offset points", ha=ha, va="center", fontsize=7, linespacing=1,
                arrowprops=dict(arrowstyle="-", lw=0.4, color="0.4", shrinkA=0, shrinkB=2.5))
ax.text(0.02, 0.04, f"ρ = {d.x.corr(d.y, method='spearman'):.2f}", transform=ax.transAxes)
h, l = ax.get_legend_handles_labels()
ax.legend(h[::-1], l[::-1], loc="upper right", bbox_to_anchor=(1, 1), fontsize=7, frameon=False,
          handlelength=0.6, handletextpad=0.3, labelspacing=0.15, borderpad=0, borderaxespad=0)
ax.set(xlim=(0, 100), xticks=[0, 25, 50, 75, 100], xlabel="Transcript assignment (%)", ylabel="Marker gene F1 score")

for i, (tool, p) in enumerate(pairs.items()):
    for prior, refined in p.items():
        af.plot([2.2 * i, 2.2 * i + 1], table.loc[[prior, refined], "marker_f1_raw"], color=grey[bool(is3d([prior])[0])],
                lw=0.8, marker="o", ms=2.5, mew=0)
af.set_xticks([0, 1, 2.2, 3.2], ["Prior", "Proseg", "Prior", "Baysor"], rotation=90)
af.set_xlim(-0.4, 3.6)
af.tick_params(axis="y", left=False)
af.spines["left"].set_visible(False)

print(d.groupby("3d").size().to_dict(), "methods")
for ext in ("png", "svg"):
    fig.savefig(out / f"assigned_vs_marker_f1.{ext}", dpi=300, bbox_inches="tight", pad_inches=.02, transparent=True)

### MECR adjusted for transcripts per cell
Raw MECR follows transcripts per cell; the score uses the residual from a log-linear fit per cohort (baselines, open circles, excluded from the fit).

In [ ]:
fig, axs = plt.subplots(1, len(cohorts), figsize=(4.5, 1.5), sharey=True, layout="constrained")
for ax, c in zip(axs, cohorts):
    t = tables[c].dropna(subset=["MECR", "tpc"])
    base = is_baseline(t.index)
    m = t[~base]
    ax.scatter(m.tpc, m.MECR, s=8, c=[grey[k] for k in is3d(m.index)], lw=0)
    ax.scatter(t.tpc[base], t.MECR[base], s=8, facecolors="none", edgecolors="0.4", lw=0.5)
    xs = np.linspace(*np.log10(m.tpc).agg(["min", "max"]), 50)
    ax.plot(10**xs, np.polyval(np.polyfit(np.log10(m.tpc), m.MECR, 1), xs), color="0.4", lw=0.6, zorder=0)
    rho = [m[y].corr(m.tpc, method="spearman") for y in ["MECR", "MECR_adj_raw"]]
    ax.text(0.97, 0.95, f"ρ = {rho[0]:.2f} → {rho[1]:.2f}", transform=ax.transAxes, ha="right", va="top", fontsize=6)
    ax.set(xscale="log", title=cohort_names[c], xlabel="Transcripts per cell")
axs[0].set_ylabel("MECR")
for ext in ("png", "svg"):
    fig.savefig(out / f"mecr_adjustment.{ext}", dpi=300, bbox_inches="tight", pad_inches=.02, transparent=True)

## All metrics (exploratory)
Table metrics from the benchmark table; extra metrics as cohort means of sample means. Baselines excluded.

In [ ]:
def read(file, query="index == index"):
    d = pd.concat(pd.read_csv(p, index_col=0).assign(cohort=c) for c in cohorts if (p := metrics_dir / c / file).exists())
    return d.query("sample not in ['all', 'all_samples']").query(query, engine="python").groupby(["cohort", "sample", "method"])


med = pd.concat([
    read("general_stats/general_stats.csv", "cell_type_revised == 'all'").mean(numeric_only=True),
    read("cell_type_metrics/clustering_score_adata_integrated_cell_type_revised.csv")[["calinski_harabasz_score"]].mean(),
    read("cell_type_metrics/cell_type_distribution_adata_integrated_cell_type_revised.csv").Undefined.mean(),
    read("marker_gene_metrics/negative_marker_purity_all.csv").negative_marker_purity.mean(),
    read("marker_gene_metrics/MECR_score_all.csv", "gene1 == 'all'").MECR.mean(),
], axis=1).groupby(["cohort", "method"]).mean().groupby("method").mean()
med = med.rename(index=clean_method_name, columns=metric_names)
med = med.drop(columns=[*vals.columns, "Z integrity", "Ovrlpy_stats_mean_integrity"], errors="ignore").join(vals, how="inner")
med = med[~is_baseline(med.index)]
seg = pd.Series(np.where(is3d(med.index), "3D", "2D"), index=med.index)
print(med.shape[0], "methods,", med.shape[1], "metrics")

In [ ]:
g = sns.clustermap(med.corr("spearman").fillna(0), cmap="RdBu_r", vmin=-1, vmax=1, annot=True, fmt=".1f", annot_kws={"size": 5},
                   figsize=(5, 5), dendrogram_ratio=0.05, cbar_pos=None, linewidths=0)
g.ax_heatmap.set(xlabel="", ylabel="")
g.ax_heatmap.tick_params(labelsize=6, length=0)
g.savefig(out / "metric_correlation_all.png", dpi=600, bbox_inches="tight", transparent=True)
for name, cols in [("all", med.columns), ("table", vals.columns)]:
    g = sns.pairplot(med[cols].assign(seg=seg), hue="seg", palette={"3D": grey[True], "2D": grey[False]}, corner=True, height=1.4, plot_kws={"s": 10, "lw": 0})
    g.savefig(out / f"metric_pairs_{name}.png", dpi=300, bbox_inches="tight", transparent=True)